# Travelling salesman problem (TSP)

**The task:** a salesman has to visit *n* cities, each exactly once, and return home. Which order of cities gives the shortest round trip?

It sounds like a puzzle, but the same task is solved every day by delivery vans, drilling machines for circuit boards,
warehouse robots and DNA-sequencing software. In this notebook we will:

1. build a small random map of cities,
2. find the best route by **trying every possibility** (brute force) and measure how the time explodes,
3. extrapolate: how many cities can we handle before the computation takes longer than the age of the Universe,
4. see a smarter exact method (**Held–Karp**) – still hopeless for large *n*,
5. use **heuristics** that give a good (not necessarily optimal) route quickly.

*How to read the code:* this is not a programming course. Every code cell has comments (`# ...`) explaining what the
lines do in plain words – read those, the exact Python syntax is secondary.

In [ ]:
# --- Libraries (ready-made tools we borrow) ---------------------------------
import matplotlib.pyplot as plt                 # drawing pictures and plots
import networkx as nx                           # working with graphs (cities = nodes, roads = edges)
import networkx.algorithms.approximation as nx_app   # ready-made heuristic algorithms for graphs (Christofides)
import math                                     # square roots, factorials, ...
from time import perf_counter                   # a precise stopwatch for measuring computation time
import numpy as np                              # tables of numbers (matrices) and fast arithmetic on them
import itertools                                # generating all permutations / combinations
import pandas as pd                             # nicely formatted tables

## 1. A random map of cities

We place cities at random points inside a unit square. The distance between two cities is the ordinary straight-line
(Euclidean) distance. All distances are stored in a **distance matrix**: a table where row *i*, column *j* holds the
distance from city *i* to city *j*. This table is the only thing the algorithms below need.

In [ ]:
def tsp_graph(cities: int = 20):
    """Create `cities` random points, return the graph, their positions and the distance matrix."""
    # random_geometric_graph gives us random points in the unit square (seed=3 -> the same map every time we run it)
    G = nx.random_geometric_graph(cities, radius=0.4, seed=3)
    pos = nx.get_node_attributes(G, "pos")      # dictionary: city number -> (x, y) coordinates

    # City 0 is the depot ("home"); we put it in the middle of the map
    pos[0] = (0.5, 0.5)

    # An empty table cities x cities that we will fill with distances
    distance_matrix = np.zeros((cities, cities))

    # For every pair of cities (i, j) with i < j ...
    for i in range(len(pos)):
        for j in range(i + 1, len(pos)):
            # ... compute the straight-line distance (Pythagoras: hypot = sqrt(dx^2 + dy^2))
            dist = math.hypot(pos[i][0] - pos[j][0], pos[i][1] - pos[j][1])
            distance_matrix[i, j] = dist        # store it in the table
            G.add_edge(i, j, weight=dist)       # and add a road i-j of that length to the graph

    # We filled only the upper half of the table; distance i->j equals j->i, so we copy it to the lower half
    distance_matrix += distance_matrix.T

    return G, pos, distance_matrix

In [ ]:
def tsp_graph_vis(G, pos, path=[]):
    """Draw the map. If a `path` (list of cities) is given, draw only that route in red; otherwise draw all roads."""
    if not path:
        # no route given -> draw all roads thin and blue
        nx.draw_networkx_edges(G, pos, edge_color="blue", width=0.5)

    # turn the route [0, 3, 1, ..., 0] into a list of consecutive pairs [(0,3), (3,1), ...]
    edge_list = list(nx.utils.pairwise(path))

    # draw cities (red circles with numbers) and the route (thick red lines)
    nx.draw_networkx(
        G,
        pos,
        with_labels=True,
        edgelist=edge_list,
        node_color='r',
        edge_color="red",
        node_size=200,
        width=3,
    )

In [ ]:
def compute_distance(distance_matrix: np.ndarray, path: list) -> tuple[int, np.ndarray]:
    """Total length of a round trip `path` (list of city numbers). The trip is closed automatically (last -> first)."""
    ind_y = path                    # cities where each leg starts:  [c0, c1, c2, ..., ck]
    ind_x = path[1:] + path[:1]     # cities where each leg ends:    [c1, c2, ..., ck, c0]  (shifted by one, wraps around)
    # distance_matrix[ind_y, ind_x] picks the length of every leg at once; .sum() adds them up
    return distance_matrix[ind_y, ind_x].sum(), distance_matrix[ind_y, ind_x]

In [ ]:
# Build a small instance with 6 cities and show the map and the distance matrix
start_point = 0
G, pos, distance_matrix = tsp_graph(6)
tsp_graph_vis(G, pos)
pd.DataFrame(distance_matrix)     # pandas shows the matrix as a readable table

## 2. Brute force: try every route

The simplest correct method: write down **every** possible order of the cities, compute the length of each,
keep the shortest. With the start fixed at city 0, the remaining n−1 cities can be ordered in (n−1)! ways
(n−1 choices for the first stop, n−2 for the second, ...). For 6 cities that is 5! = 120 routes – easy.
We will soon see what happens for 15 or 20 cities.

In [ ]:
def brute_force_tsp(distance_matrix: np.ndarray, start: int) -> tuple[list[int], float]:
    """Try all orderings of the cities and return the shortest route and the time it took."""
    points = list(range(distance_matrix.shape[0]))   # all city numbers [0, 1, ..., n-1]
    points.remove(start)                              # the start is fixed, we permute only the others

    best_distance = np.inf          # "infinity": any real route will be shorter than this
    best_permutation = []

    start_time = perf_counter()     # start the stopwatch

    # itertools.permutations(points) produces every possible ordering of the remaining cities, one by one
    for perm in itertools.permutations(points):
        distance = distance_matrix[start, perm[0]]              # leg from home to the first city
        for i in range(len(perm) - 1):
            distance += distance_matrix[perm[i], perm[i + 1]]   # legs between consecutive cities
        distance += distance_matrix[perm[-1], start]            # leg from the last city back home

        if distance < best_distance:                            # found a shorter route -> remember it
            best_distance = distance
            best_permutation = [start] + list(perm) + [start]

    compute_time = perf_counter() - start_time                  # stop the stopwatch

    return best_permutation, compute_time

In [ ]:
path, compute_time = brute_force_tsp(distance_matrix, 0)
print("Optimal path: ", path, "Distance: ", compute_distance(distance_matrix, path)[0], "Time: ", compute_time, 's')
tsp_graph_vis(G, pos, path)
plt.title(f'Optimal, distance: {compute_distance(distance_matrix, path)[0]}')

### How does the time grow with the number of cities?
We run brute force for 2, 3, ..., 11 cities and measure each run. (11 cities = 10! = 3 628 800 routes, a few seconds.)

In [ ]:
times = []                                   # here we collect the measured times
for i in range(2, 12):                       # i = 2, 3, ..., 11 cities
    _, _, distance_matrix_test = tsp_graph(i)              # a map with i cities
    _, compute_time = brute_force_tsp(distance_matrix_test, 0)
    print(f'Cities: {i}, computation time: {compute_time} s')
    times.append(compute_time)
plt.plot(range(2, 12), times)                # linear plot: "nothing, nothing, nothing, wall"
plt.xlabel("number of cities"); plt.ylabel("time [s]")

## 3. How far can brute force go? Extrapolation to the age of the Universe

The measured times grow like (n−1)! – the number of routes we have to evaluate.
Let us estimate the time for **one route evaluation** from the largest measured instance
and extrapolate: how many cities can we handle before the computation takes longer than
the time elapsed since the **Big Bang** (≈ 13.8 billion years)?

In [ ]:
brute_force_times = dict(zip(range(2, 12), times))   # measurements from the cell above: {2: t2, 3: t3, ..., 11: t11}

YEAR = 365.25 * 24 * 3600                 # seconds in a year
AGE_OF_UNIVERSE = 13.8e9 * YEAR           # ≈ 4.35e17 seconds since the Big Bang


def human_time(seconds: float) -> str:
    """Convert a number of seconds into a readable text: '3.2 ms', '1.7 days', '6.4 thousand years', ..."""
    units = [("ns", 1e-9), ("µs", 1e-6), ("ms", 1e-3), ("s", 1), ("min", 60), ("h", 3600),
             ("days", 86400), ("years", YEAR), ("thousand years", 1e3 * YEAR),
             ("million years", 1e6 * YEAR), ("billion years", 1e9 * YEAR)]
    if seconds > 1e6 * 1e9 * YEAR:                       # absurdly large -> scientific notation
        return f"{seconds / (1e9 * YEAR):.1e} billion years"
    for name, size in reversed(units):                   # find the largest unit that fits
        if seconds >= size:
            return f"{seconds / size:,.1f} {name}"
    return f"{seconds / 1e-9:.2f} ns"


# Time per one evaluated route = (time for 11 cities) / (number of routes for 11 cities)
n_measured = max(brute_force_times)
time_per_route = brute_force_times[n_measured] / math.factorial(n_measured - 1)
print(f"Measured: {n_measured} cities -> {math.factorial(n_measured - 1):,} routes in {brute_force_times[n_measured]:.2f} s")
print(f"=> one route evaluation takes ~{time_per_route * 1e9:.0f} ns\n")

# Now estimate the time for n = 2, 3, 4, ... until it exceeds the age of the Universe
rows = []
n = 2
while True:
    routes = math.factorial(n - 1)             # (n-1)! routes
    estimate = routes * time_per_route         # estimated seconds
    rows.append({"cities": n,
                 "routes (n-1)!": f"{routes:.3e}",
                 "estimated time": human_time(estimate),
                 "ages of the Universe": f"{estimate / AGE_OF_UNIVERSE:.2e}",
                 "seconds": estimate})
    if estimate > AGE_OF_UNIVERSE:             # stop when we pass the age of the Universe
        break
    n += 1

extrapolation = pd.DataFrame(rows).set_index("cities")
print(f"Brute force exceeds the age of the Universe at n = {n} cities.")
extrapolation.drop(columns="seconds")

In [ ]:
# The same numbers as a plot. The vertical axis is LOGARITHMIC: each tick is 1000x more.
plt.figure(figsize=(8, 5))
plt.semilogy(extrapolation.index, extrapolation["seconds"], "o-", label="brute force (n-1)! – extrapolated")
plt.semilogy(list(brute_force_times), list(brute_force_times.values()), "s", color="red", label="measured")

# Horizontal reference lines so that the numbers mean something
references = {"1 second": 1, "1 hour": 3600, "1 year": YEAR,
              "human life (80 y)": 80 * YEAR, "age of the Universe": AGE_OF_UNIVERSE}
for label, value in references.items():
    plt.axhline(value, ls="--", color="gray", lw=0.8)
    plt.text(2, value * 1.5, label, fontsize=8, color="gray")

plt.xlabel("number of cities")
plt.ylabel("computation time [s]")
plt.title("Brute force TSP: computation time vs. number of cities")
plt.grid(True, which="both", alpha=0.3)
plt.legend()
plt.show()

### Would a faster computer help?

Let us assume we get a computer that is 1000×, 1 000 000× or 1 000 000 000× faster
(the last one is roughly *all computers on Earth working together*).
How many more cities can we solve within an hour, a year, or the age of the Universe?

In [ ]:
def max_cities(time_budget: float, speedup: float = 1.0) -> int:
    """Largest number of cities whose brute force solution fits into `time_budget` seconds on a `speedup`x faster machine."""
    n = 2
    # keep adding cities while the estimated time still fits into the budget  ((n+1) cities -> n! routes)
    while math.factorial(n) * time_per_route / speedup <= time_budget:
        n += 1
    return n

budgets = {"1 hour": 3600, "1 year": YEAR, "age of the Universe": AGE_OF_UNIVERSE}
speedups = [1, 1e3, 1e6, 1e9]

table = pd.DataFrame({f"{s:,.0f}x faster": [max_cities(b, s) for b in budgets.values()] for s in speedups},
                     index=budgets.keys())
table.index.name = "time budget"
print("Maximum number of cities solvable by brute force:")
table

## 4. Exact, but smarter: dynamic programming (Held–Karp)

Brute force repeats a lot of work: the routes A-B-C-D-E and A-B-C-E-D share the beginning A-B-C, yet it is summed twice.
Held–Karp remembers, for every *set of already visited cities* and every *last city*, the shortest way to get there –
and never computes the same thing twice. Its complexity is O(n²·2ⁿ) – **still exponential**, but 2ⁿ grows much slower
than n!. It is the best known exact algorithm for the general TSP.

Note: nobody knows a polynomial algorithm for TSP – and if you found one, you would have proven P = NP
(and earned \$1 000 000 – see the `P_vs_NP` notebook).

In [ ]:
def held_karp_tsp(distance_matrix: np.ndarray, start: int) -> tuple[list[int], float]:
    """Exact TSP by dynamic programming over subsets of cities."""
    n = distance_matrix.shape[0]
    others = [c for c in range(n) if c != start]   # all cities except the start
    m = len(others)

    start_time = perf_counter()

    # dp[(mask, j)] = length of the shortest path from `start` that visits exactly the cities in `mask` and ends in others[j].
    # `mask` is a set of cities written as a binary number: bit j = 1 means "others[j] has been visited".
    dp, parent = {}, {}
    for j in range(m):
        dp[(1 << j, j)] = distance_matrix[start, others[j]]     # one visited city: just the road from the start

    for mask in range(1, 1 << m):                # go through all 2^m subsets (all binary numbers with m bits)
        for j in range(m):                       # j = the city where the path currently ends
            if not (mask >> j) & 1 or (mask, j) not in dp:      # j must be in the set and the entry must exist
                continue
            base = dp[(mask, j)]
            for k in range(m):                   # try extending the path to a not-yet-visited city k
                if (mask >> k) & 1:              # k already visited -> skip
                    continue
                new_mask = mask | (1 << k)       # the set with k added
                candidate = base + distance_matrix[others[j], others[k]]
                if candidate < dp.get((new_mask, k), np.inf):   # better than what we had -> store it
                    dp[(new_mask, k)] = candidate
                    parent[(new_mask, k)] = j    # remember where we came from (needed to rebuild the route)

    full = (1 << m) - 1                          # the set with all cities (all bits = 1)
    # close the tour: add the road back to the start and take the best last city
    best_j = min(range(m), key=lambda j: dp[(full, j)] + distance_matrix[others[j], start])

    # Rebuild the route by following the `parent` pointers backwards
    path, mask, j = [], full, best_j
    while True:
        path.append(others[j])
        previous = parent.get((mask, j))
        if previous is None:
            break
        mask ^= 1 << j                           # remove j from the set
        j = previous
    path = [start] + path[::-1] + [start]        # reverse and add the start at both ends

    compute_time = perf_counter() - start_time
    return path, compute_time


path, compute_time = held_karp_tsp(distance_matrix, 0)
print("Optimal path: ", path, "Distance: ", compute_distance(distance_matrix, path)[0], "Time: ", compute_time, 's')
tsp_graph_vis(G, pos, path)
plt.title(f'Held-Karp (optimal), distance: {compute_distance(distance_matrix, path)[0]}')

In [ ]:
# Held-Karp complexity: measured up to 15 cities, compared with brute force
held_karp_times = {}
for i in range(3, 16):
    _, _, distance_matrix_test = tsp_graph(i)
    _, compute_time = held_karp_tsp(distance_matrix_test, 0)
    held_karp_times[i] = compute_time
    print(f'Cities: {i}, computation time: {compute_time:.4f} s')

# time per elementary operation of the DP, estimated from the largest instance (n^2 * 2^n operations)
n_hk = max(held_karp_times)
time_per_op = held_karp_times[n_hk] / (n_hk ** 2 * 2 ** n_hk)

def brute_force_estimate(n): return math.factorial(int(n) - 1) * time_per_route
def held_karp_estimate(n):   return float(n) ** 2 * 2.0 ** n * time_per_op

n_hk_universe = next(n for n in range(3, 200) if held_karp_estimate(n) > AGE_OF_UNIVERSE)
print(f"\nHeld-Karp exceeds the age of the Universe at n = {n_hk_universe} cities (brute force: n = {len(extrapolation) + 1}).")

sizes = [10, 15, 20, 25, 30, 40, 50, 60, 70]
comparison = pd.DataFrame({
    "brute force": [human_time(brute_force_estimate(n)) for n in sizes],
    "Held-Karp":   [human_time(held_karp_estimate(n))   for n in sizes],
}, index=pd.Index(sizes, name="cities"))

ns = np.arange(2, 80)
plt.figure(figsize=(8, 5))
plt.semilogy(ns, [brute_force_estimate(n) for n in ns], label="brute force  (n-1)!")
plt.semilogy(ns, [held_karp_estimate(n) for n in ns], label="Held-Karp  n^2 * 2^n")
plt.semilogy(list(brute_force_times), list(brute_force_times.values()), "s", color="C0")
plt.semilogy(list(held_karp_times), list(held_karp_times.values()), "s", color="C1")
plt.axhline(AGE_OF_UNIVERSE, ls="--", color="gray"); plt.text(3, AGE_OF_UNIVERSE * 2, "age of the Universe", color="gray")
plt.axhline(3600, ls="--", color="gray"); plt.text(3, 3600 * 2, "1 hour", color="gray")
plt.ylim(1e-6, 1e30)
plt.xlabel("number of cities"); plt.ylabel("computation time [s]"); plt.grid(True, which="both", alpha=0.3); plt.legend()
plt.title("Exact TSP algorithms (squares = measured, lines = extrapolated)")
plt.show()
comparison

**Conclusion:** exact algorithms stop working somewhere between 20 and 60 cities – a parcel delivery van visits more stops every day.
This is why in practice we use **heuristics** – fast algorithms that give a *good* (not necessarily optimal) tour.
Below are three classic ones.

## 5. Heuristic 1: nearest neighbour (greedy)

"From the current city always go to the nearest city you have not visited yet." Extremely fast (about n² operations),
no guarantee – the last legs are often long because the remaining cities are far away.

In [ ]:
def greedy_tsp(distance_matrix: np.ndarray, start: int) -> tuple[list[int], float]:
    """Nearest-neighbour heuristic."""
    points = list(range(distance_matrix.shape[0]))   # cities still to visit
    points.remove(start)

    visited = [start]               # the route we are building
    next_point = start              # where we currently stand

    start_time = perf_counter()

    while points:                                          # while there are unvisited cities
        distances = distance_matrix[next_point, points]    # distances from the current city to all unvisited ones
        min_distance_index = np.argmin(distances)          # position of the smallest one
        next_point = points.pop(min_distance_index)        # go there (and remove it from the unvisited list)
        visited.append(next_point)

    visited.append(start)           # return home

    compute_time = perf_counter() - start_time

    return visited, compute_time

In [ ]:
path, compute_time = greedy_tsp(distance_matrix, 0)
print("Path: ", path, "Distance: ", compute_distance(distance_matrix, path)[0], "Time: ", compute_time, 's')
tsp_graph_vis(G, pos, path)
plt.title(f'Greedy, distance: {compute_distance(distance_matrix, path)[0]}')

In [ ]:
# Greedy complexity: 2..29 cities – all in a fraction of a millisecond
times = []
for i in range(2, 30):
    _, _, distance_matrix_test = tsp_graph(i)
    _, compute_time = greedy_tsp(distance_matrix_test, 0)
    print(f'Cities: {i}, computation time: {compute_time} s')
    times.append(compute_time)
plt.plot(range(2, 30), times)
plt.xlabel("number of cities"); plt.ylabel("time [s]")

## 6. Heuristic 2: minimum spanning tree (MST) – guaranteed at most 2× optimum

1. Build the **minimum spanning tree**: the cheapest set of roads that connects all cities (no cycles).
2. Walk around the tree (depth-first) and write down the cities in the order of their first visit.
3. That order is the route. Skipping already-visited cities never makes it longer (triangle inequality).

Why at most 2× optimum: the optimal tour minus one road is a spanning tree, so MST ≤ OPT; walking around the tree
uses each road twice, so the walk is ≤ 2·MST ≤ 2·OPT; shortcuts only shorten it.

In [ ]:
def mst_approx_tsp(distance_matrix: np.ndarray, start: int) -> tuple[list[int], float]:
    """MST-based 2-approximation."""
    import numpy as np
    from scipy.sparse.csgraph import minimum_spanning_tree   # ready-made MST algorithm (Kruskal/Prim inside)
    from collections import defaultdict

    num_cities = distance_matrix.shape[0]

    start_time = perf_counter()

    # Step 1: compute the minimum spanning tree (result: a matrix with the tree's roads only)
    mst_sparse = minimum_spanning_tree(distance_matrix)
    mst_matrix = mst_sparse.toarray()
    mst_matrix += mst_matrix.T          # make it symmetric (roads go both ways)

    # Turn the matrix into a "who is connected to whom" list
    adjacency_list = defaultdict(list)
    for i in range(num_cities):
        for j in range(num_cities):
            if mst_matrix[i, j] > 0:
                adjacency_list[i].append(j)

    # Step 2: depth-first walk around the tree, recording the first visit of each city
    visited = []
    visited_set = set()

    def dfs(u):
        visited.append(u)               # first visit of u -> it goes into the route
        visited_set.add(u)
        for v in adjacency_list[u]:     # then visit all neighbours in the tree ...
            if v not in visited_set:    # ... that we have not seen yet
                dfs(v)

    dfs(start)
    visited.append(start)               # return home  (step 3 - the shortcuts - happens automatically:
                                        #  we only recorded first visits, so the "way back" is skipped)

    compute_time = perf_counter() - start_time

    return visited, compute_time

In [ ]:
path, compute_time = mst_approx_tsp(distance_matrix, 0)
print("Path: ", path, "Distance: ", compute_distance(distance_matrix, path)[0], "Time: ", compute_time, 's')
tsp_graph_vis(G, pos, path)
plt.title(f'MST, distance: {compute_distance(distance_matrix, path)[0]}')

In [ ]:
# MST heuristic complexity: 2..29 cities
times = []
for i in range(2, 30):
    _, _, distance_matrix_test = tsp_graph(i)
    _, compute_time = mst_approx_tsp(distance_matrix_test, 0)
    print(f'Cities: {i}, computation time: {compute_time} s')
    times.append(compute_time)
plt.plot(range(2, 30), times)
plt.xlabel("number of cities"); plt.ylabel("time [s]")

## 7. Heuristic 3: Christofides – guaranteed at most 1.5× optimum

MST + the cheapest pairing of the tree's odd-degree cities + an Euler circuit (every road once) + shortcuts.
We use the ready-made implementation from networkx. For 44 years (1976–2020) this was the best known guarantee.

In [ ]:
start_time = perf_counter()
path = nx_app.christofides(G, weight="weight")      # networkx does all the steps for us
compute_time = perf_counter() - start_time
print("Path: ", path, "Distance: ", compute_distance(distance_matrix, path)[0], "Time: ", compute_time, 's')
tsp_graph_vis(G, pos, path)
plt.title(f'Christofides, distance: {compute_distance(distance_matrix, path)[0]}')

## 8. All three heuristics on 20 cities
Brute force would need 19!/2 ≈ 6·10¹⁶ routes here – impossible. The heuristics answer in milliseconds.
Compare the three pictures: look for crossing roads (a sign that the route can still be improved by "2-opt").

In [ ]:
cities = 20
G, pos, distance_matrix = tsp_graph(cities)
tsp_graph_vis(G, pos)
plt.title('Travelling salesman problem')

path, compute_time = greedy_tsp(distance_matrix, 0)
plt.figure()
tsp_graph_vis(G, pos, path)
plt.title(f'Greedy, distance: {compute_distance(distance_matrix, path)[0]:.3f}')

path, compute_time = mst_approx_tsp(distance_matrix, 0)
plt.figure()
tsp_graph_vis(G, pos, path)
plt.title(f'MST, distance: {compute_distance(distance_matrix, path)[0]:.3f}')

path = nx_app.christofides(G, weight="weight")
plt.figure()
tsp_graph_vis(G, pos, path)
plt.title(f'Christofides, distance: {compute_distance(distance_matrix, path)[0]:.3f}')

## Exercise

Implement **2-opt**: repeatedly look for two roads (a,b) and (c,d) in the route such that replacing them by (a,c) and (b,d)
makes the route shorter (that means reversing the part of the route between b and c). Stop when no such pair exists.
Start from the greedy route and compare the result with Christofides on 20 and 100 cities.

In [ ]:
def two_opt(distance_matrix: np.ndarray, path: list[int]) -> list[int]:
    """Improve a closed route by 2-opt moves. `path` starts and ends with the same city."""
    route = path[:-1]               # work without the repeated start at the end
    improved = True
    while improved:
        improved = False
        # YOUR CODE HERE:
        #   for every pair of positions i < k:
        #       a, b = route[i], route[i + 1];  c, d = route[k], route[(k + 1) % len(route)]
        #       if d(a,c) + d(b,d) < d(a,b) + d(c,d):  reverse route[i+1 : k+1]  and set improved = True
        pass
    return route + [route[0]]

# path, _ = greedy_tsp(distance_matrix, 0)
# better = two_opt(distance_matrix, path)
# print("greedy:", compute_distance(distance_matrix, path)[0], " after 2-opt:", compute_distance(distance_matrix, better)[0])